## Enfoque de transformación orientado a modelo estrella

Como continuidad del trabajo desarrollado en la Semana 1, en esta práctica se toma la información extraída desde las tres fuentes seleccionadas y se aplica la fase de transformación del proceso ETL.

Además de limpiar, normalizar y validar los datos, se prepara una estructura analítica orientada a un modelo estrella. Este enfoque permite organizar los datos transformados en una tabla de hechos y varias dimensiones, facilitando futuras consultas, análisis e integración con PostgreSQL.

El modelo estrella propuesto se centra en las observaciones de señales WiFi CSI. La tabla de hechos almacena medidas derivadas de amplitud, fase y estadísticas CSI, mientras que las dimensiones permiten describir la fuente de datos, actividad, usuario, objeto, posición, configuración y tiempo.

Es importante aclarar que las fuentes originales no poseen una clave común directa para unir todos los registros. Por esta razón, el modelo estrella se construye a partir de los datos transformados y no mediante una unión directa de las fuentes originales.


# Práctica Semana 2 - Transformación ETL orientada a modelo estrella

## Grupo 10

## Por
- Manuel Vicente

## Integrantes
- Manuel Vicente
- Elvis Borbor
- Klever Barahona

## Continuidad con Semana 1

En la Semana 1 se realizó la identificación de fuentes de datos, organización del proyecto, configuración de Docker, uso de PostgreSQL y definición de las fuentes relacionadas con WiFi CSI.

En esta Semana 2 se continúa con la fase de transformación del proceso ETL. Se aplican procesos de exploración, limpieza, normalización, selección de columnas, generación de variables derivadas e identificadores.

Como resultado adicional, se prepara una estructura analítica tipo modelo estrella para facilitar futuras cargas y consultas en PostgreSQL.

## Configuración segura del proyecto

El proyecto utiliza variables de entorno almacenadas en un archivo `.env` para evitar escribir credenciales directamente en el notebook.

Esta configuración se conecta con el trabajo realizado en la Semana 1, donde se definió una base de datos PostgreSQL ejecutada mediante Docker.

El uso de `.env` permite separar la configuración sensible del código fuente y facilita la portabilidad del proyecto.

In [ ]:
from pathlib import Path
import os
import json
import zipfile

import pandas as pd
import numpy as np
from dotenv import load_dotenv

In [32]:
load_dotenv()

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
POSTGRES_DATA_DIR = DATA_DIR / "postgres"
JSON_DIR = DATA_DIR / "json"
HAR_ORT_DIR = DATA_DIR / "har_ort"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")

print("Proyecto:", PROJECT_ROOT)
print("Base de datos:", POSTGRES_DB)
print("Puerto:", POSTGRES_PORT)

Proyecto: c:\Users\Manuel\Desktop\proyectoWifi
Base de datos: wifi_csi_db
Puerto: 5433


## Infraestructura Docker y PostgreSQL

El proyecto utiliza Docker para levantar un servicio PostgreSQL 16. Esta base de datos funcionará como destino para las tablas transformadas y para la estructura analítica tipo modelo estrella.

El archivo `docker-compose.yml` define el servicio `postgres`, el contenedor `postgres_wifi_csi`, el volumen persistente y el puerto local `5433`.

Para iniciar el servicio se ejecuta:

```bash
docker compose up -d


---

## Fase 4. Carga de fuentes originales

Aquí mantienes las tres fuentes.

### Fuente 1

```python
ruta_object_csv = POSTGRES_DATA_DIR / "raw_data.csv"

df_object = pd.read_csv(ruta_object_csv)

print("Object Detection:", df_object.shape)
df_object.head()

In [33]:
json_files = list(JSON_DIR.rglob("*.json"))

for file in json_files:
    print(file)

c:\Users\Manuel\Desktop\proyectoWifi\data\json\data_apml_min.json


In [34]:
ruta_har_json = json_files[0]

with open(ruta_har_json, "r") as file:
    data_har = json.load(file)

print("Cantidad de muestras:", len(data_har))
print("Dimensión primera muestra:", np.array(data_har[0]).shape)

Cantidad de muestras: 21298
Dimensión primera muestra: (9, 56)


In [35]:
registros_har = []

for i, muestra in enumerate(data_har):
    matriz = np.array(muestra)

    registros_har.append({
        "sample_id_original": i + 1,
        "csi_mean": matriz.mean(),
        "csi_min": matriz.min(),
        "csi_max": matriz.max(),
        "csi_std": matriz.std(),
        "csi_median": np.median(matriz),
        "csi_range": matriz.max() - matriz.min(),
        "num_rows_matrix": matriz.shape[0],
        "num_columns_matrix": matriz.shape[1]
    })

df_har = pd.DataFrame(registros_har)

print("HAR transformado desde JSON:", df_har.shape)
df_har.head()

HAR transformado desde JSON: (21298, 9)


,sample_id_original,csi_mean,csi_min,csi_max,csi_std,csi_median,csi_range,num_rows_matrix,num_columns_matrix
0,1,92.915050,17.464249,176.139150,41.452590,77.729468,158.674900,9,56
1,2,87.472571,26.683328,165.423698,33.616029,79.918708,138.740370,9,56
2,3,94.036743,23.086793,172.699160,38.146070,96.992149,149.612368,9,56
3,4,99.896125,36.055513,166.207701,31.917030,104.472354,130.152189,9,56
4,5,110.250978,40.049969,184.390889,34.661883,108.353575,144.340920,9,56


In [36]:
annotation_files = list(HAR_ORT_DIR.rglob("annotations.csv"))

for file in annotation_files:
    print(file)

c:\Users\Manuel\Desktop\proyectoWifi\data\har_ort\annotations.csv


In [37]:
ruta_annotations = annotation_files[0]

df_annotations = pd.read_csv(ruta_annotations)

print("Annotations:", df_annotations.shape)
df_annotations.head()

Annotations: (1527, 4)


,id,activity,user,filename
0,1,fall,benja,fall_benja_2025-07-12_00-23-38.pcap
1,2,fall,benja,fall_benja_2025-07-12_00-23-47.pcap
2,3,fall,benja,fall_benja_2025-07-12_00-23-56.pcap
3,4,fall,benja,fall_benja_2025-07-12_00-24-05.pcap
4,5,fall,benja,fall_benja_2025-07-12_00-24-14.pcap


## Exploración inicial

Antes de transformar los datos se revisa la estructura de cada fuente. Esta fase permite identificar tipos de datos, valores nulos, duplicados, columnas relevantes y posibles problemas de formato.

Esta exploración es necesaria para decidir qué columnas se conservan, qué transformaciones se aplican y cómo se prepararán los datos para el modelo estrella.

In [38]:
def explorar_dataframe(df, nombre):
    print("=" * 80)
    print(f"Exploración: {nombre}")
    print("=" * 80)

    print("\nFilas y columnas:")
    print(df.shape)

    print("\nTipos de datos:")
    print(df.dtypes)

    print("\nValores nulos:")
    print(df.isnull().sum())

    print("\nDuplicados:")
    print(df.duplicated().sum())

    display(df.head())

In [39]:
explorar_dataframe(df_object, "WiFi CSI Object Detection")
explorar_dataframe(df_har, "WiFi CSI HAR")
explorar_dataframe(df_annotations, "Annotations HAR-ORT")

Exploración: WiFi CSI Object Detection

Filas y columnas:
(56939, 109)

Tipos de datos:
amp_0            float64
amp_1            float64
amp_2            float64
amp_3            float64
amp_4            float64
                  ...   
type                 str
day                int64
object_id          int64
position             str
configuration        str
Length: 109, dtype: object

Valores nulos:
amp_0            0
amp_1            0
amp_2            0
amp_3            0
amp_4            0
                ..
type             0
day              0
object_id        0
position         0
configuration    0
Length: 109, dtype: int64

Duplicados:
0


,amp_0,amp_1,amp_2,amp_3,amp_4,amp_5,amp_6,amp_7,amp_8,amp_9,amp_10,amp_11,amp_12,amp_13,amp_14,amp_15,amp_16,amp_17,amp_18,amp_19,amp_20,amp_21,amp_22,amp_23,amp_24,amp_25,amp_26,amp_27,amp_28,amp_29,amp_30,amp_31,amp_32,amp_33,amp_34,amp_35,amp_36,amp_37,amp_38,amp_39,amp_40,amp_41,amp_42,amp_43,amp_44,amp_45,amp_46,amp_47,amp_48,amp_49,amp_50,amp_51,phase_0,phase_1,phase_2,phase_3,phase_4,phase_5,phase_6,phase_7,phase_8,phase_9,phase_10,phase_11,phase_12,phase_13,phase_14,phase_15,phase_16,phase_17,phase_18,phase_19,phase_20,phase_21,phase_22,phase_23,phase_24,phase_25,phase_26,phase_27,phase_28,phase_29,phase_30,phase_31,phase_32,phase_33,phase_34,phase_35,phase_36,phase_37,phase_38,phase_39,phase_40,phase_41,phase_42,phase_43,phase_44,phase_45,phase_46,phase_47,phase_48,phase_49,phase_50,phase_51,type,day,object_id,position,configuration
0,14.317821,15.524175,14.317821,15.524175,15.524175,14.866069,16.155494,14.866069,16.552945,15.652476,16.643317,16.643317,16.401219,15.811388,15.000000,14.422205,13.892444,13.038405,13.601471,13.038405,13.038405,13.038405,13.038405,13.038405,13.038405,13.416408,13.416408,13.000000,13.000000,13.928388,12.083046,13.416408,13.416408,12.529964,13.416408,11.704700,13.000000,12.083046,11.704700,11.704700,12.165525,13.038405,13.152946,12.165525,12.165525,13.038405,13.152946,14.142136,13.152946,14.317821,14.317821,12.165525,1.781890,1.831399,1.781890,1.831399,1.831399,1.913820,1.951303,1.913820,2.007423,2.034444,2.142134,2.142134,2.226492,2.176341,2.214297,2.158799,2.098871,2.137526,2.199593,2.137526,2.137526,2.137526,2.137526,2.137526,2.137526,2.034444,2.034444,1.965587,1.965587,1.937970,1.997424,2.034444,2.034444,2.070143,2.034444,1.919567,1.965587,1.997424,1.919567,1.919567,1.735945,1.647568,1.723446,1.735945,1.735945,1.647568,1.723446,1.712693,1.723446,1.781890,1.781890,1.735945,organic,1,0,A10,COMPRIDO
1,19.209373,19.798990,18.439089,21.931712,20.518285,19.798990,20.518285,21.400935,22.203603,22.203603,21.633308,21.633308,20.248457,20.248457,19.723083,20.248457,18.357560,18.027756,18.357560,17.492856,18.357560,17.492856,16.643317,15.620499,17.804494,16.278821,18.439089,17.029386,16.970563,17.691806,17.029386,17.029386,17.029386,16.401219,12.727922,11.313708,15.000000,15.811388,16.401219,15.811388,14.422205,16.124515,15.652476,15.811388,16.552945,15.231546,16.552945,18.788294,16.155494,16.155494,16.552945,15.652476,2.245537,2.356194,2.279423,2.323948,2.321725,2.356194,2.390664,2.488746,2.516108,2.516108,2.553590,2.553590,2.567288,2.567288,2.609869,2.567288,2.629203,2.553590,2.629203,2.601173,2.629203,2.601173,2.570255,2.446854,2.475623,2.399645,2.432966,2.439336,2.356194,2.396173,2.439336,2.439336,2.273053,2.485897,2.356194,2.356194,2.214297,2.176341,2.226492,2.176341,2.158799,2.089942,2.034444,1.892547,2.007423,1.975688,2.007423,2.010639,1.951303,1.951303,2.007423,2.034444,organic,1,0,A10,COMPRIDO
2,15.132746,15.132746,14.142136,15.132746,16.124515,15.033296,16.000000,16.031220,16.000000,16.031220,16.278821,16.124515,15.297059,15.297059,13.601471,15.132746,14.317821,13.341664,13.341664,13.152946,13.152946,13.152946,12.165525,13.038405,12.041595,12.041595,12.041595,12.041595,13.038405,12.165525,13.038405,13.038405,13.038405,13.000000,13.038405,12.041595,12.041595,11.045361,11.180340,11.401754,11.704700,10.770330,11.704700,12.083046,12.083046,12.083046,13.000000,13.416408,13.000000,13.416408,12.529964,12.083046,1.438245,1.438245,1.428899,1.438245,1.446441,1.504228,1.570796,1.633215,1.570796,1.633215,1.756144,1.695151,1.768192,1.768192,1.869295,1.703348,1.781890,1.797595,1.797595,1.723446,1.723446,1.723446,1.735945,1.647568,1.653938,1.653938,1.653938,1.487655,1.494024,1.405648,1.494024,1.494024,1.647568,1.570796,1.494024,1.487655,1.487655,1.480136,1.390943,1.304544,1.222025,1.190290,1.222025,1.144169,1.144169,1.144169,1.176005,1.107149,1.176005,1.107149,1.071450,1.144169,organic,1,0,A10,COMPRIDO
3,12.649111,13.928388,13.000000,13.928388,14.317821,14.317821,15.652476,15.264338,15.264338,14.76482

Exploración: WiFi CSI HAR

Filas y columnas:
(21298, 9)

Tipos de datos:
sample_id_original      int64
csi_mean              float64
csi_min               float64
csi_max               float64
csi_std               float64
csi_median            float64
csi_range             float64
num_rows_matrix         int64
num_columns_matrix      int64
dtype: object

Valores nulos:
sample_id_original    0
csi_mean              0
csi_min               0
csi_max               0
csi_std               0
csi_median            0
csi_range             0
num_rows_matrix       0
num_columns_matrix    0
dtype: int64

Duplicados:
0


,sample_id_original,csi_mean,csi_min,csi_max,csi_std,csi_median,csi_range,num_rows_matrix,num_columns_matrix
0,1,92.915050,17.464249,176.139150,41.452590,77.729468,158.674900,9,56
1,2,87.472571,26.683328,165.423698,33.616029,79.918708,138.740370,9,56
2,3,94.036743,23.086793,172.699160,38.146070,96.992149,149.612368,9,56
3,4,99.896125,36.055513,166.207701,31.917030,104.472354,130.152189,9,56
4,5,110.250978,40.049969,184.390889,34.661883,108.353575,144.340920,9,56


Exploración: Annotations HAR-ORT

Filas y columnas:
(1527, 4)

Tipos de datos:
id          int64
activity      str
user          str
filename      str
dtype: object

Valores nulos:
id          0
activity    0
user        0
filename    0
dtype: int64

Duplicados:
0


,id,activity,user,filename
0,1,fall,benja,fall_benja_2025-07-12_00-23-38.pcap
1,2,fall,benja,fall_benja_2025-07-12_00-23-47.pcap
2,3,fall,benja,fall_benja_2025-07-12_00-23-56.pcap
3,4,fall,benja,fall_benja_2025-07-12_00-24-05.pcap
4,5,fall,benja,fall_benja_2025-07-12_00-24-14.pcap


## Limpieza general

Se crean funciones reutilizables para limpiar los DataFrames. Estas funciones normalizan nombres de columnas, limpian textos y eliminan registros duplicados.

Esto permite mantener un criterio homogéneo de limpieza antes de construir las dimensiones y la tabla de hechos.

In [ ]:
def limpiar_nombres_columnas(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def normalizar_textos(df):
    df = df.copy()
    columnas_texto = df.select_dtypes(include="object").columns

    for col in columnas_texto:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )

    return df


def limpiar_dataframe(df):
    df = df.copy()
    df = limpiar_nombres_columnas(df)
    df = normalizar_textos(df)
    df = df.drop_duplicates()
    return df

In [40]:
df_object_clean = limpiar_dataframe(df_object)
df_har_clean = limpiar_dataframe(df_har)
df_annotations_clean = limpiar_dataframe(df_annotations)

C:\Users\Manuel\AppData\Local\Temp\ipykernel_19052\3546967330.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df.select_dtypes(include="object").columns
C:\Users\Manuel\AppData\Local\Temp\ipykernel_19052\3546967330.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

In [41]:
amp_cols = [col for col in df_object_clean.columns if col.startswith("amp_")]
phase_cols = [col for col in df_object_clean.columns if col.startswith("phase_")]

print("Columnas amplitud:", len(amp_cols))
print("Columnas fase:", len(phase_cols))

Columnas amplitud: 52
Columnas fase: 52


In [42]:
for col in amp_cols + phase_cols:
    df_object_clean[col] = pd.to_numeric(df_object_clean[col], errors="coerce")

In [43]:
df_object_fact_base = df_object_clean.copy()

df_object_fact_base["amp_mean"] = df_object_fact_base[amp_cols].mean(axis=1)
df_object_fact_base["amp_std"] = df_object_fact_base[amp_cols].std(axis=1)
df_object_fact_base["amp_min"] = df_object_fact_base[amp_cols].min(axis=1)
df_object_fact_base["amp_max"] = df_object_fact_base[amp_cols].max(axis=1)
df_object_fact_base["amp_range"] = df_object_fact_base["amp_max"] - df_object_fact_base["amp_min"]

df_object_fact_base["phase_mean"] = df_object_fact_base[phase_cols].mean(axis=1)
df_object_fact_base["phase_std"] = df_object_fact_base[phase_cols].std(axis=1)
df_object_fact_base["phase_min"] = df_object_fact_base[phase_cols].min(axis=1)
df_object_fact_base["phase_max"] = df_object_fact_base[phase_cols].max(axis=1)
df_object_fact_base["phase_range"] = df_object_fact_base["phase_max"] - df_object_fact_base["phase_min"]

df_object_fact_base["has_object"] = df_object_fact_base["type"].apply(
    lambda x: 0 if x == "empty" else 1
)

df_object_fact_base["source_name"] = "wifi_csi_object_detection"

df_object_fact_base.head()

,amp_0,amp_1,amp_2,amp_3,amp_4,amp_5,amp_6,amp_7,amp_8,amp_9,amp_10,amp_11,amp_12,amp_13,amp_14,amp_15,amp_16,amp_17,amp_18,amp_19,amp_20,amp_21,amp_22,amp_23,amp_24,amp_25,amp_26,amp_27,amp_28,amp_29,amp_30,amp_31,amp_32,amp_33,amp_34,amp_35,amp_36,amp_37,amp_38,amp_39,amp_40,amp_41,amp_42,amp_43,amp_44,amp_45,amp_46,amp_47,amp_48,amp_49,amp_50,amp_51,phase_0,phase_1,phase_2,phase_3,phase_4,phase_5,phase_6,phase_7,...,phase_9,phase_10,phase_11,phase_12,phase_13,phase_14,phase_15,phase_16,phase_17,phase_18,phase_19,phase_20,phase_21,phase_22,phase_23,phase_24,phase_25,phase_26,phase_27,phase_28,phase_29,phase_30,phase_31,phase_32,phase_33,phase_34,phase_35,phase_36,phase_37,phase_38,phase_39,phase_40,phase_41,phase_42,phase_43,phase_44,phase_45,phase_46,phase_47,phase_48,phase_49,phase_50,phase_51,type,day,object_id,position,configuration,amp_mean,amp_std,amp_min,amp_max,amp_range,phase_mean,phase_std,phase_min,phase_max,phase_range,has_object,source_name
0,14.317821,15.524175,14.317821,15.524175,15.524175,14.866069,16.155494,14.866069,16.552945,15.652476,16.643317,16.643317,16.401219,15.811388,15.000000,14.422205,13.892444,13.038405,13.601471,13.038405,13.038405,13.038405,13.038405,13.038405,13.038405,13.416408,13.416408,13.000000,13.000000,13.928388,12.083046,13.416408,13.416408,12.529964,13.416408,11.704700,13.000000,12.083046,11.704700,11.704700,12.165525,13.038405,13.152946,12.165525,12.165525,13.038405,13.152946,14.142136,13.152946,14.317821,14.317821,12.165525,1.781890,1.831399,1.781890,1.831399,1.831399,1.913820,1.951303,1.913820,...,2.034444,2.142134,2.142134,2.226492,2.176341,2.214297,2.158799,2.098871,2.137526,2.199593,2.137526,2.137526,2.137526,2.137526,2.137526,2.137526,2.034444,2.034444,1.965587,1.965587,1.937970,1.997424,2.034444,2.034444,2.070143,2.034444,1.919567,1.965587,1.997424,1.919567,1.919567,1.735945,1.647568,1.723446,1.735945,1.735945,1.647568,1.723446,1.712693,1.723446,1.781890,1.781890,1.735945,organic,1,0,a10,comprido,13.765029,1.387948,11.704700,16.643317,4.938617,1.956086,0.168423,1.647568,2.226492,0.578924,1,wifi_csi_object_detection
1,19.209373,19.798990,18.439089,21.931712,20.518285,19.798990,20.518285,21.400935,22.203603,22.203603,21.633308,21.633308,20.248457,20.248457,19.723083,20.248457,18.357560,18.027756,18.357560,17.492856,18.357560,17.492856,16.643317,15.620499,17.804494,16.278821,18.439089,17.029386,16.970563,17.691806,17.029386,17.029386,17.029386,16.401219,12.727922,11.313708,15.000000,15.811388,16.401219,15.811388,14.422205,16.124515,15.652476,15.811388,16.552945,15.231546,16.552945,18.788294,16.155494,16.155494,16.552945,15.652476,2.245537,2.356194,2.279423,2.323948,2.321725,2.356194,2.390664,2.488746,...,2.516108,2.553590,2.553590,2.567288,2.567288,2.609869,2.567288,2.629203,2.553590,2.629203,2.601173,2.629203,2.601173,2.570255,2.446854,2.475623,2.399645,2.432966,2.439336,2.356194,2.396173,2.439336,2.439336,2.273053,2.485897,2.356194,2.356194,2.214297,2.176341,2.226492,2.176341,2.158799,2.089942,2.034444,1.892547,2.007423,1.975688,2.007423,2.010639,1.951303,1.951303,2.007423,2.034444,organic,1,0,a10,comprido,17.740957,2.386393,11.313708,22.203603,10.889895,2.339212,0.219022,1.892547,2.629203,0.736656,1,wifi_csi_object_detection
2,15.132746,15.132746,14.142136,15.132746,16.124515,15.033296,16.000000,16.031220,16.000000,16.031220,16.278821,16.124515,15.297059,15.297059,13.601471,15.132746,14.317821,13.341664,13.341664,13.152946,13.152946,13.152946,12.165525,13.038405,12.041595,12.041595,12.041595,12.041595,13.038405,12.165525,13.038405,13.038405,13.038405,13.000000,13.038405,12.041595,12.041595,11.045361,11.180340,11.401754,11.704700,10.770330,11.704700,12.083046,12.083046,12.083046,13.000000,13.416408,13.000000,13.416408,12.529964,12.083046,1.438245,1.438245,1.428899,1.438245,1.446441,1.504228,1.570796,1.633215,...,1.633215,1.756144,1.695151,1.768192,1.768192,1.869295,1.703348,1.781890,1.797595,1.797595,1.723446,1.723446,1.723446,1.735945,1.647568,1.653938,1.653938,1.653938,1.48765

In [44]:
df_har_fact_base = df_har_clean.copy()

df_har_fact_base["source_name"] = "wifi_csi_human_activity_recognition"

df_har_fact_base["csi_level"] = pd.cut(
    df_har_fact_base["csi_mean"],
    bins=3,
    labels=["bajo", "medio", "alto"]
)

df_har_fact_base["variability_level"] = pd.cut(
    df_har_fact_base["csi_std"],
    bins=3,
    labels=["baja", "media", "alta"]
)

df_har_fact_base.head()

,sample_id_original,csi_mean,csi_min,csi_max,csi_std,csi_median,csi_range,num_rows_matrix,num_columns_matrix,source_name,csi_level,variability_level
0,1,92.915050,17.464249,176.139150,41.452590,77.729468,158.674900,9,56,wifi_csi_human_activity_recognition,medio,baja
1,2,87.472571,26.683328,165.423698,33.616029,79.918708,138.740370,9,56,wifi_csi_human_activity_recognition,medio,baja
2,3,94.036743,23.086793,172.699160,38.146070,96.992149,149.612368,9,56,wifi_csi_human_activity_recognition,medio,baja
3,4,99.896125,36.055513,166.207701,31.917030,104.472354,130.152189,9,56,wifi_csi_human_activity_recognition,medio,baja
4,5,110.250978,40.049969,184.390889,34.661883,108.353575,144.340920,9,56,wifi_csi_human_activity_recognition,medio,baja


In [45]:
df_annotations_clean["filename"] = df_annotations_clean["filename"].astype(str).str.strip().str.lower()

patron_filename = r"^(?P<activity_file>[^_]+)_(?P<user_file>[^_]+)_(?P<capture_date>\d{4}-\d{2}-\d{2})_(?P<capture_time>\d{2}-\d{2}-\d{2})\.pcap$"

datos_extraidos = df_annotations_clean["filename"].str.extract(patron_filename)

df_annotations_fact_base = pd.concat([df_annotations_clean, datos_extraidos], axis=1)

df_annotations_fact_base["capture_time"] = df_annotations_fact_base["capture_time"].str.replace("-", ":", regex=False)

df_annotations_fact_base["capture_datetime"] = pd.to_datetime(
    df_annotations_fact_base["capture_date"] + " " + df_annotations_fact_base["capture_time"],
    errors="coerce"
)

df_annotations_fact_base["source_name"] = "har_ort_annotations"

df_annotations_fact_base.head()

,id,activity,user,filename,activity_file,user_file,capture_date,capture_time,capture_datetime,source_name
0,1,fall,benja,fall_benja_2025-07-12_00-23-38.pcap,fall,benja,2025-07-12,00:23:38,2025-07-12 00:23:38,har_ort_annotations
1,2,fall,benja,fall_benja_2025-07-12_00-23-47.pcap,fall,benja,2025-07-12,00:23:47,2025-07-12 00:23:47,har_ort_annotations
2,3,fall,benja,fall_benja_2025-07-12_00-23-56.pcap,fall,benja,2025-07-12,00:23:56,2025-07-12 00:23:56,har_ort_annotations
3,4,fall,benja,fall_benja_2025-07-12_00-24-05.pcap,fall,benja,2025-07-12,00:24:05,2025-07-12 00:24:05,har_ort_annotations
4,5,fall,benja,fall_benja_2025-07-12_00-24-14.pcap,fall,benja,2025-07-12,00:24:14,2025-07-12 00:24:14,har_ort_annotations


## Construcción de dimensiones

En un modelo estrella, las dimensiones contienen información descriptiva que permite analizar la tabla de hechos desde diferentes perspectivas.

Para este proyecto se crean dimensiones de fuente, actividad, usuario, objeto, posición, configuración y tiempo.

In [46]:
dim_source = pd.DataFrame({
    "source_name": [
        "wifi_csi_object_detection",
        "wifi_csi_human_activity_recognition",
        "har_ort_annotations"
    ]
})

dim_source.insert(0, "source_id", range(1, len(dim_source) + 1))

dim_source

,source_id,source_name
0,1,wifi_csi_object_detection
1,2,wifi_csi_human_activity_recognition
2,3,har_ort_annotations


In [47]:
activities = df_annotations_fact_base[["activity"]].drop_duplicates().copy()
activities = activities.rename(columns={"activity": "activity_name"})
activities = activities.dropna()

dim_activity = activities.reset_index(drop=True)
dim_activity.insert(0, "activity_id", range(1, len(dim_activity) + 1))

dim_activity

,activity_id,activity_name
0,1,fall
1,2,quiet
2,3,sit
3,4,stand
4,5,walk


In [48]:
dim_activity = pd.concat([
    pd.DataFrame({"activity_id": [0], "activity_name": ["no_aplica"]}),
    dim_activity
], ignore_index=True)

dim_activity

,activity_id,activity_name
0,0,no_aplica
1,1,fall
2,2,quiet
3,3,sit
4,4,stand
5,5,walk


In [49]:
users = df_annotations_fact_base[["user"]].drop_duplicates().copy()
users = users.rename(columns={"user": "user_name"})
users = users.dropna()

dim_user = users.reset_index(drop=True)
dim_user.insert(0, "user_id", range(1, len(dim_user) + 1))

dim_user = pd.concat([
    pd.DataFrame({"user_id": [0], "user_name": ["no_aplica"]}),
    dim_user
], ignore_index=True)

dim_user

,user_id,user_name
0,0,no_aplica
1,1,benja
2,2,bruno
3,3,flor
4,4,luigi
5,5,mathias
6,6,pablo
7,7,paula
8,8,peke
9,9,vero


In [50]:
objects = df_object_fact_base[["object_id", "type"]].drop_duplicates().copy()
objects = objects.rename(columns={"type": "object_type"})

dim_object = objects.reset_index(drop=True)
dim_object.insert(0, "object_dim_id", range(1, len(dim_object) + 1))

dim_object = pd.concat([
    pd.DataFrame({
        "object_dim_id": [0],
        "object_id": ["no_aplica"],
        "object_type": ["no_aplica"]
    }),
    dim_object
], ignore_index=True)

dim_object.head()

,object_dim_id,object_id,object_type
0,0,no_aplica,no_aplica
1,1,0,organic
2,2,1,organic
3,3,2,metalic
4,4,3,metalic


In [51]:
positions = df_object_fact_base[["position"]].drop_duplicates().copy()
positions = positions.dropna()

dim_position = positions.reset_index(drop=True)
dim_position.insert(0, "position_id", range(1, len(dim_position) + 1))

dim_position = pd.concat([
    pd.DataFrame({"position_id": [0], "position": ["no_aplica"]}),
    dim_position
], ignore_index=True)

dim_position

,position_id,position
0,0,no_aplica
1,1,a10
2,2,a16
3,3,b21
4,4,c14
5,5,c19
6,6,c6
7,7,d10
8,8,e17
9,9,f12


In [52]:
configurations = df_object_fact_base[["configuration"]].drop_duplicates().copy()
configurations = configurations.dropna()

dim_configuration = configurations.reset_index(drop=True)
dim_configuration.insert(0, "configuration_id", range(1, len(dim_configuration) + 1))

dim_configuration = pd.concat([
    pd.DataFrame({"configuration_id": [0], "configuration": ["no_aplica"]}),
    dim_configuration
], ignore_index=True)

dim_configuration

,configuration_id,configuration
0,0,no_aplica
1,1,comprido
2,2,curto


In [53]:
time_data = df_annotations_fact_base[["capture_datetime"]].dropna().drop_duplicates().copy()

time_data["date"] = time_data["capture_datetime"].dt.date
time_data["year"] = time_data["capture_datetime"].dt.year
time_data["month"] = time_data["capture_datetime"].dt.month
time_data["day"] = time_data["capture_datetime"].dt.day
time_data["hour"] = time_data["capture_datetime"].dt.hour

dim_time = time_data.reset_index(drop=True)
dim_time.insert(0, "time_id", range(1, len(dim_time) + 1))

dim_time = pd.concat([
    pd.DataFrame({
        "time_id": [0],
        "capture_datetime": [pd.NaT],
        "date": [pd.NaT],
        "year": [0],
        "month": [0],
        "day": [0],
        "hour": [0]
    }),
    dim_time
], ignore_index=True)

dim_time.head()

,time_id,capture_datetime,date,year,month,day,hour
0,0,NaT,NaT,0,0,0,0
1,1,2025-07-12 00:23:38,2025-07-12,2025,7,12,0
2,2,2025-07-12 00:23:47,2025-07-12,2025,7,12,0
3,3,2025-07-12 00:23:56,2025-07-12,2025,7,12,0
4,4,2025-07-12 00:24:05,2025-07-12,2025,7,12,0


## Construcción de la tabla de hechos

La tabla de hechos almacena las medidas principales del análisis. En este caso, las medidas corresponden a estadísticas derivadas de señales WiFi CSI, amplitud, fase y variables de control.

Cada registro de la tabla de hechos se relaciona con dimensiones mediante identificadores numéricos.

In [54]:
fact_object = df_object_fact_base.copy()

fact_object = fact_object.merge(dim_source, on="source_name", how="left")
fact_object = fact_object.merge(dim_object, on=["object_id", "object_type"] if "object_type" in fact_object.columns else ["object_id"], how="left")

In [55]:
fact_object = df_object_fact_base.copy()

fact_object = fact_object.merge(dim_source, on="source_name", how="left")

fact_object = fact_object.merge(
    dim_object,
    left_on=["object_id", "type"],
    right_on=["object_id", "object_type"],
    how="left"
)

fact_object = fact_object.merge(dim_position, on="position", how="left")
fact_object = fact_object.merge(dim_configuration, on="configuration", how="left")

fact_object["activity_id"] = 0
fact_object["user_id"] = 0
fact_object["time_id"] = 0

fact_object_star = fact_object[[
    "source_id",
    "activity_id",
    "user_id",
    "object_dim_id",
    "position_id",
    "configuration_id",
    "time_id",
    "amp_mean",
    "amp_std",
    "amp_min",
    "amp_max",
    "amp_range",
    "phase_mean",
    "phase_std",
    "phase_min",
    "phase_max",
    "phase_range",
    "has_object"
]].copy()

fact_object_star.head()

,source_id,activity_id,user_id,object_dim_id,position_id,configuration_id,time_id,amp_mean,amp_std,amp_min,amp_max,amp_range,phase_mean,phase_std,phase_min,phase_max,phase_range,has_object
0,1,0,0,1,1,1,0,13.765029,1.387948,11.704700,16.643317,4.938617,1.956086,0.168423,1.647568,2.226492,0.578924,1
1,1,0,0,1,1,1,0,17.740957,2.386393,11.313708,22.203603,10.889895,2.339212,0.219022,1.892547,2.629203,0.736656,1
2,1,0,0,1,1,1,0,13.389721,1.560541,10.770330,16.278821,5.508491,1.493132,0.225293,1.071450,1.869295,0.797846,1
3,1,0,0,1,1,1,0,12.915181,1.402389,10.440307,16.401219,5.960913,0.441101,0.209186,0.000000,0.785398,0.785398,1
4,1,0,0,1,1,1,0,18.055356,1.996149,14.866069,22.090722,7.224653,1.173832,2.712283,-3.096169,3.141593,6.237762,1


In [56]:
fact_har = df_har_fact_base.copy()

fact_har = fact_har.merge(dim_source, on="source_name", how="left")

fact_har["activity_id"] = 0
fact_har["user_id"] = 0
fact_har["object_dim_id"] = 0
fact_har["position_id"] = 0
fact_har["configuration_id"] = 0
fact_har["time_id"] = 0

fact_har["amp_mean"] = np.nan
fact_har["amp_std"] = np.nan
fact_har["amp_min"] = np.nan
fact_har["amp_max"] = np.nan
fact_har["amp_range"] = np.nan

fact_har["phase_mean"] = np.nan
fact_har["phase_std"] = np.nan
fact_har["phase_min"] = np.nan
fact_har["phase_max"] = np.nan
fact_har["phase_range"] = np.nan

fact_har["has_object"] = np.nan

fact_har_star = fact_har[[
    "source_id",
    "activity_id",
    "user_id",
    "object_dim_id",
    "position_id",
    "configuration_id",
    "time_id",
    "amp_mean",
    "amp_std",
    "amp_min",
    "amp_max",
    "amp_range",
    "phase_mean",
    "phase_std",
    "phase_min",
    "phase_max",
    "phase_range",
    "csi_mean",
    "csi_std",
    "csi_min",
    "csi_max",
    "csi_range",
    "has_object"
]].copy()

fact_har_star.head()

,source_id,activity_id,user_id,object_dim_id,position_id,configuration_id,time_id,amp_mean,amp_std,amp_min,amp_max,amp_range,phase_mean,phase_std,phase_min,phase_max,phase_range,csi_mean,csi_std,csi_min,csi_max,csi_range,has_object
0,2,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92.915050,41.452590,17.464249,176.139150,158.674900,NaN
1,2,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.472571,33.616029,26.683328,165.423698,138.740370,NaN
2,2,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,94.036743,38.146070,23.086793,172.699160,149.612368,NaN
3,2,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,99.896125,31.917030,36.055513,166.207701,130.152189,NaN
4,2,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,110.250978,34.661883,40.049969,184.390889,144.340920,NaN


El archivo `annotations.csv` se utiliza principalmente para alimentar dimensiones de actividad, usuario y tiempo. No se incluye como hecho principal porque no contiene medidas numéricas de señales CSI como amplitud, fase o estadísticas de señal.

In [57]:
fact_object_star["csi_mean"] = np.nan
fact_object_star["csi_std"] = np.nan
fact_object_star["csi_min"] = np.nan
fact_object_star["csi_max"] = np.nan
fact_object_star["csi_range"] = np.nan

fact_object_star = fact_object_star[fact_har_star.columns]

fact_wifi_csi_observation = pd.concat(
    [fact_object_star, fact_har_star],
    ignore_index=True
)

fact_wifi_csi_observation.insert(
    0,
    "fact_id",
    range(1, len(fact_wifi_csi_observation) + 1)
)

fact_wifi_csi_observation.head()

,fact_id,source_id,activity_id,user_id,object_dim_id,position_id,configuration_id,time_id,amp_mean,amp_std,amp_min,amp_max,amp_range,phase_mean,phase_std,phase_min,phase_max,phase_range,csi_mean,csi_std,csi_min,csi_max,csi_range,has_object
0,1,1,0,0,1,1,1,0,13.765029,1.387948,11.704700,16.643317,4.938617,1.956086,0.168423,1.647568,2.226492,0.578924,NaN,NaN,NaN,NaN,NaN,1.0
1,2,1,0,0,1,1,1,0,17.740957,2.386393,11.313708,22.203603,10.889895,2.339212,0.219022,1.892547,2.629203,0.736656,NaN,NaN,NaN,NaN,NaN,1.0
2,3,1,0,0,1,1,1,0,13.389721,1.560541,10.770330,16.278821,5.508491,1.493132,0.225293,1.071450,1.869295,0.797846,NaN,NaN,NaN,NaN,NaN,1.0
3,4,1,0,0,1,1,1,0,12.915181,1.402389,10.440307,16.401219,5.960913,0.441101,0.209186,0.000000,0.785398,0.785398,NaN,NaN,NaN,NaN,NaN,1.0
4,5,1,0,0,1,1,1,0,18.055356,1.996149,14.866069,22.090722,7.224653,1.173832,2.712283,-3.096169,3.141593,6.237762,NaN,NaN,NaN,NaN,NaN,1.0


## Validación del modelo estrella

Se valida que la tabla de hechos tenga identificadores, medidas y claves hacia dimensiones. También se revisa que las dimensiones no tengan duplicados y que la tabla de hechos mantenga consistencia básica.

In [58]:
print("Dim source:", dim_source.shape)
print("Dim activity:", dim_activity.shape)
print("Dim user:", dim_user.shape)
print("Dim object:", dim_object.shape)
print("Dim position:", dim_position.shape)
print("Dim configuration:", dim_configuration.shape)
print("Dim time:", dim_time.shape)
print("Fact:", fact_wifi_csi_observation.shape)

Dim source: (3, 2)
Dim activity: (6, 2)
Dim user: (12, 2)
Dim object: (7, 3)
Dim position: (12, 2)
Dim configuration: (3, 2)
Dim time: (1528, 7)
Fact: (78237, 24)


In [59]:
print("Duplicados dim_source:", dim_source.duplicated().sum())
print("Duplicados dim_activity:", dim_activity.duplicated().sum())
print("Duplicados dim_user:", dim_user.duplicated().sum())
print("Duplicados dim_object:", dim_object.duplicated().sum())
print("Duplicados dim_position:", dim_position.duplicated().sum())
print("Duplicados dim_configuration:", dim_configuration.duplicated().sum())
print("Duplicados dim_time:", dim_time.duplicated().sum())

Duplicados dim_source: 0
Duplicados dim_activity: 0
Duplicados dim_user: 0
Duplicados dim_object: 0
Duplicados dim_position: 0
Duplicados dim_configuration: 0
Duplicados dim_time: 0


In [60]:
fact_wifi_csi_observation.isnull().sum()

fact_id                 0
source_id               0
activity_id             0
user_id                 0
object_dim_id           0
position_id             0
configuration_id        0
time_id                 0
amp_mean            21298
amp_std             21298
amp_min             21298
amp_max             21298
amp_range           21298
phase_mean          21298
phase_std           21298
phase_min           21298
phase_max           21298
phase_range         21298
csi_mean            56939
csi_std             56939
csi_min             56939
csi_max             56939
csi_range           56939
has_object          21298
dtype: int64

In [61]:
dim_source.to_csv(PROCESSED_DIR / "dim_source.csv", index=False)
dim_activity.to_csv(PROCESSED_DIR / "dim_activity.csv", index=False)
dim_user.to_csv(PROCESSED_DIR / "dim_user.csv", index=False)
dim_object.to_csv(PROCESSED_DIR / "dim_object.csv", index=False)
dim_position.to_csv(PROCESSED_DIR / "dim_position.csv", index=False)
dim_configuration.to_csv(PROCESSED_DIR / "dim_configuration.csv", index=False)
dim_time.to_csv(PROCESSED_DIR / "dim_time.csv", index=False)

fact_wifi_csi_observation.to_csv(
    PROCESSED_DIR / "fact_wifi_csi_observation.csv",
    index=False
)

print("Modelo estrella exportado correctamente.")

Modelo estrella exportado correctamente.


In [62]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [63]:
db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    host="localhost",
    port=POSTGRES_PORT,
    database=POSTGRES_DB
)

engine = create_engine(db_url)

In [64]:
dim_source.to_sql("dim_source", engine, if_exists="replace", index=False)
dim_activity.to_sql("dim_activity", engine, if_exists="replace", index=False)
dim_user.to_sql("dim_user", engine, if_exists="replace", index=False)
dim_object.to_sql("dim_object", engine, if_exists="replace", index=False)
dim_position.to_sql("dim_position", engine, if_exists="replace", index=False)
dim_configuration.to_sql("dim_configuration", engine, if_exists="replace", index=False)
dim_time.to_sql("dim_time", engine, if_exists="replace", index=False)

fact_wifi_csi_observation.to_sql(
    "fact_wifi_csi_observation",
    engine,
    if_exists="replace",
    index=False
)

print("Modelo estrella cargado en PostgreSQL.")

Modelo estrella cargado en PostgreSQL.


In [65]:
with engine.connect() as conn:
    tablas = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """))

    for tabla in tablas:
        print(tabla[0])

dim_activity
dim_configuration
dim_object
dim_position
dim_source
dim_time
dim_user
fact_wifi_csi_observation
wifi_csi_object_detection


## Validación de tablas cargadas en PostgreSQL

Después de cargar las dimensiones y la tabla de hechos en PostgreSQL, se valida la cantidad de registros almacenados en cada tabla.

Esta validación permite comprobar que el modelo estrella fue cargado correctamente y que las tablas están disponibles para futuras consultas.

In [66]:
tablas_modelo = [
    "dim_source",
    "dim_activity",
    "dim_user",
    "dim_object",
    "dim_position",
    "dim_configuration",
    "dim_time",
    "fact_wifi_csi_observation"
]

with engine.connect() as conn:
    for tabla in tablas_modelo:
        total = conn.execute(
            text(f"SELECT COUNT(*) FROM {tabla};")
        ).scalar()
        print(f"{tabla}: {total} registros")

dim_source: 3 registros
dim_activity: 6 registros
dim_user: 12 registros
dim_object: 7 registros
dim_position: 12 registros
dim_configuration: 3 registros
dim_time: 1528 registros
fact_wifi_csi_observation: 78237 registros


## Consulta de evidencia del modelo estrella

Se realiza una consulta sobre la tabla de hechos y la dimensión fuente para comprobar que la estructura del modelo estrella permite analizar los registros por fuente de datos.

Esta consulta demuestra que la tabla de hechos se puede relacionar con sus dimensiones.

In [68]:
query_columnas_fact = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'fact_wifi_csi_observation'
ORDER BY ordinal_position;
"""

with engine.connect() as conn:
    columnas_fact_postgres = pd.read_sql(query_columnas_fact, conn)

columnas_fact_postgres

,column_name,data_type
0,fact_id,bigint
1,source_id,bigint
2,activity_id,bigint
3,user_id,bigint
4,object_dim_id,bigint
5,position_id,bigint
6,configuration_id,bigint
7,time_id,bigint
8,amp_mean,double precision
9,amp_std,double precision


In [69]:
query_resumen_fuente = """
SELECT 
    ds.source_name,
    COUNT(*) AS total_registros
FROM fact_wifi_csi_observation f
INNER JOIN dim_source ds
    ON f.source_id = ds.source_id
GROUP BY ds.source_name
ORDER BY total_registros DESC;
"""

with engine.connect() as conn:
    resumen_postgres = pd.read_sql(query_resumen_fuente, conn)

resumen_postgres

,source_name,total_registros
0,wifi_csi_object_detection,56939
1,wifi_csi_human_activity_recognition,21298


## Evidencia de consulta por fuente

Se realizó una consulta entre la tabla de hechos `fact_wifi_csi_observation` y la dimensión `dim_source`.

Esta consulta permite validar que los registros transformados fueron cargados correctamente en PostgreSQL y que la tabla de hechos puede relacionarse con sus dimensiones mediante claves numéricas.

In [70]:
query_actividades = """
SELECT 
    da.activity_name,
    COUNT(*) AS total_registros
FROM fact_wifi_csi_observation f
INNER JOIN dim_activity da
    ON f.activity_id = da.activity_id
GROUP BY da.activity_name
ORDER BY total_registros DESC;
"""

with engine.connect() as conn:
    resumen_actividades_postgres = pd.read_sql(query_actividades, conn)

resumen_actividades_postgres

,activity_name,total_registros
0,no_aplica,78237


## Evidencia de consulta por actividad

Se realizó una consulta entre la tabla de hechos y la dimensión `dim_activity`.

Esta consulta permite validar que las actividades provenientes del archivo `annotations.csv` fueron integradas dentro del modelo estrella mediante claves dimensionales.

In [71]:
query_objetos = """
SELECT 
    do2.object_type,
    COUNT(*) AS total_registros,
    AVG(f.amp_mean) AS promedio_amplitud,
    AVG(f.phase_mean) AS promedio_fase
FROM fact_wifi_csi_observation f
INNER JOIN dim_object do2
    ON f.object_dim_id = do2.object_dim_id
GROUP BY do2.object_type
ORDER BY total_registros DESC;
"""

with engine.connect() as conn:
    resumen_objetos_postgres = pd.read_sql(query_objetos, conn)

resumen_objetos_postgres

,object_type,total_registros,promedio_amplitud,promedio_fase
0,empty,27966,16.815302,-0.000684
1,no_aplica,21298,NaN,NaN
2,organic,16983,14.356316,-0.005232
3,metalic,11990,14.994178,0.003930


## Evidencia de consulta por objeto

Se consultó la tabla de hechos junto con la dimensión `dim_object` para analizar los registros por tipo de objeto.

Además, se calcularon promedios de amplitud y fase, demostrando que las medidas transformadas pueden ser utilizadas en consultas analíticas.

In [72]:
tablas_modelo = [
    "dim_source",
    "dim_activity",
    "dim_user",
    "dim_object",
    "dim_position",
    "dim_configuration",
    "dim_time",
    "fact_wifi_csi_observation"
]

with engine.connect() as conn:
    for tabla in tablas_modelo:
        total = conn.execute(
            text(f"SELECT COUNT(*) FROM {tabla};")
        ).scalar()
        print(f"{tabla}: {total} registros")

dim_source: 3 registros
dim_activity: 6 registros
dim_user: 12 registros
dim_object: 7 registros
dim_position: 12 registros
dim_configuration: 3 registros
dim_time: 1528 registros
fact_wifi_csi_observation: 78237 registros


## Validación de registros cargados en PostgreSQL

Se validó la cantidad de registros almacenados en cada dimensión y en la tabla de hechos.

Esta revisión permite confirmar que el modelo estrella fue cargado correctamente en PostgreSQL y que las tablas están disponibles para futuras consultas.

## Validación de estructura de la tabla de hechos

Se revisó la estructura de la tabla `fact_wifi_csi_observation` en PostgreSQL.

La tabla contiene claves dimensionales como `source_id`, `activity_id`, `user_id`, `object_dim_id`, `position_id`, `configuration_id` y `time_id`.

También contiene medidas numéricas derivadas de la transformación, como estadísticas de amplitud, fase y CSI. Esto confirma que la tabla de hechos conserva una estructura adecuada para un modelo estrella.

## Conclusión general

En esta práctica se continuó el trabajo desarrollado en la Semana 1, utilizando las tres fuentes seleccionadas para el proyecto WiFi CSI.

Se aplicó la fase de transformación del proceso ETL mediante exploración, limpieza, normalización, selección de columnas relevantes, creación de nuevas variables, generación de identificadores y validación de resultados.

Además, se preparó una estructura analítica tipo modelo estrella, compuesta por dimensiones de fuente, actividad, usuario, objeto, posición, configuración y tiempo, junto con una tabla de hechos denominada `fact_wifi_csi_observation`.

La estructura final fue cargada en PostgreSQL mediante Docker, permitiendo validar que los datos transformados pueden almacenarse en un sistema de destino y utilizarse para futuras consultas analíticas.

## Reflexión individual - Manuel Vicente

Esta semana aprendí que la transformación de datos dentro de un proceso ETL no solo consiste en limpiar información, sino en preparar una estructura útil para el análisis. En esta práctica comprendí cómo, a partir de datos crudos, se puede construir un modelo más organizado, como un modelo estrella, que facilite consultas y toma de decisiones.

En mi trabajo actual relaciono este aprendizaje con datos provenientes de herramientas DLP, como navegación web, uso de aplicaciones, acceso a documentos y archivos que contienen información confidencial. Estos datos pueden ayudar a construir un flujo de información más claro sobre el comportamiento de los usuarios y los posibles riesgos.

Uno de los principales problemas al trabajar con datos exportados es que suelen incluir muchas columnas que no aportan valor, por lo que la limpieza permite obtener información más eficiente y relevante. También es común encontrar problemas con fechas y zonas horarias, especialmente en herramientas alojadas en la nube.

La transformación más útil para mí fue cargar los datos en PostgreSQL, porque permite organizar la información para futuras consultas. Documentar el proceso en Markdown es fundamental, ya que deja evidencia clara de cada paso realizado.
